# 📦 Data Shipping 101: A Beginner's Guide to Apache Thrift

Welcome! If you've never heard of "Data Encoding" or "RPC," don't worry. Think of this notebook as a guide to running an **International Courier Service**.

### The Big Idea: The Language Barrier
Imagine you have a **Chinese Chef (a Python program)** and a **German Customer (a Java program)**. They need to exchange recipes, but they don't speak the same language, and sending huge, heavy books (like slow JSON files) is too expensive.

**Apache Thrift** solves this by:
1.  **Providing a Contract**: A standard "order form" everyone understands.
2.  **Vacuum Packing**: Compressing the data into a tiny binary format that's super cheap and fast to ship.

## 1. What is Apache Thrift?

Thrift is a system used to define data types and services. 

It supports basic types like:
- `bool`: True/False
- `i32/i64`: Whole numbers (integers)
- `string`: Text
- `list`: A collection of items

We define these in a `.thrift` file, which acts as our **Contract**.

## 2. Step 1: The Contract (The Schema)

Before we ship anything, we need to agree on what a "Person" looks like. 

**The Analogy: The Menu Numbers**  
In a restaurant, instead of saying "I want a double-patty beef burger with extra cheese and no onions," you just say **"I'll take a Number 1."** It's much faster. 

In the code below, notice the numbers `1:`, `2:`, and `3:`. These are our **Field Tags** (Menu Numbers). Thrift uses these numbers to identify data instead of using long names, making it incredibly fast.

In [ ]:
%%writefile ../schema/person.thrift

struct Person {
  1: required string userName,
  2: optional i64 favoriteNumber,
  3: optional list<string> interests
}

service School {
    Person teachCourse(1: required Person person, 2: required string course)
}

## 3. Step 2: The Kitchen (The Server)

Now we need someone to actually *do* something with the data. 

**The Analogy: The Chef**  
The Server is like a Chef in a kitchen. They sit there waiting for an order (an RPC call). When an order comes in, they follow the instructions and send the result back.

In our code, the `teachCourse` method is the Chef's skill: they take a person, add a new interest to their list, and return the updated person.

In [ ]:
%%writefile ../person_thrift_server.py
import thriftpy2
person_thrift = thriftpy2.load("./schema/person.thrift", module_name="person_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def teachCourse(self, person, course):
        # The 'Chef' adds a new skill to the person's interests
        person.interests.append(course)
        return person

server = make_server(person_thrift.School, School(), client_timeout=None)
print("Chef is in the kitchen (Server started)...")
print("Press Ctrl+C to stop the server.")
server.serve()

### How to run the Kitchen

1. Use the cell below to copy the command and paste on a new terminal.
2. Make sure your terminal are in the folder `5m-data-2.3-data-encoding-creation-flow`
3. Make sure you are in the right environment using `conda activate bde`.

Then, run `python person_thrift_server.py` in a new terminal. This will start the server. 

Once you see "Chef is in the kitchen," come back here to place your order!

## 4. Step 3: The Customer (The Client)

Now we act as the customer. We will create a person and send them to the kitchen to learn something new.

**The Analogy: Placing the Order**  
When you call `school.teachCourse(...)`, it feels like you're doing it locally on your computer. But behind the scenes, Thrift is **packing** your data into a box, **shipping** it to the server, and **bringing back** the result.

In [ ]:
import thriftpy2
person_thrift = thriftpy2.load("../schema/person.thrift", module_name="person_thrift")

from thriftpy2.rpc import make_client

# Connect to the kitchen
school = make_client(person_thrift.School, timeout=None)

In [ ]:
# Create a person named Martin
martin = person_thrift.Person(
    userName="Martin", favoriteNumber=1337, interests=["daydreaming", "hacking"]
)
print(f"Initial interests: {martin.interests}")

In [ ]:
# Send Martin to school to learn 'coding' remotely!
martin = school.teachCourse(martin, "coding")

In [ ]:
# Let's see if he learned it
print(f"Updated interests: {martin.interests}")

## (Optional) 🎓 Exercises

Try these to see if you've mastered the "Courier Service":

1.  **Update the Contract**: Go back to the `.thrift` cell. Add `4: optional i32 grade` to the Person struct. 
2.  **Add a Skill**: Add a new method `assignGrade` to the School service. 
3.  **Update the Chef**: Update the Server code to handle the new `assignGrade` command.
4.  **Restart**: Remember, if you change the contract, the Chef and the Customer both need to re-read it! Stop your terminal and start the server again.

**Sample solution**

**1) Update the Thrift contract**

In the .thrift cell, change Person and School to:

In [ ]:
struct Person {
  1: required string userName,
  2: optional i64 favoriteNumber,
  3: optional list<string> interests,
  4: optional i32 grade
}

service School {
    Person teachCourse(1: required Person person, 2: required string course)
    Person assignGrade(1: required Person person, 2: required i32 grade)
}

**2) Update the server**

In person_thrift_server.py, add assignGrade:

In [ ]:
import thriftpy2
person_thrift = thriftpy2.load("./schema/person.thrift", module_name="person_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def teachCourse(self, person, course):
        person.interests.append(course)
        return person

    def assignGrade(self, person, grade):
        person.grade = grade
        return person

server = make_server(person_thrift.School, School(), client_timeout=None)
print("Chef is in the kitchen (Server started)...")
print("Press Ctrl+C to stop the server.")
server.serve()

**3) Use the new method from the client**

After restarting the server, call assignGrade from the client side:

In [ ]:
import thriftpy2
person_thrift = thriftpy2.load("../schema/person.thrift", module_name="person_thrift")

from thriftpy2.rpc import make_client

school = make_client(person_thrift.School, timeout=None)

martin = person_thrift.Person(
    userName="Martin", favoriteNumber=1337, interests=["daydreaming", "hacking"]
)

# teachCourse example
martin = school.teachCourse(martin, "coding")
print(f"Interests after class: {martin.interests}")

# assignGrade example
martin = school.assignGrade(martin, 95)
print(f"Assigned grade: {martin.grade}")

**4)Important note**

After changing the contract, stop the server, restart it, and re-run the notebook cells that load person.thrift so both client and server share the updated schema.